In [ ]:
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

In [20]:
df = pd.read_csv("data/processed_crop_data.csv")

df_models = df.groupby("Model_ID")

for model_id, group in df_models:
    print(f"Training model for Model_ID: {model_id}")
    print(group[["Area_Ha", "Yield_QHa_Clipped"]].describe())

Training model for Model_ID: Model_1_UltraLow
             Area_Ha  Yield_QHa_Clipped
count   14072.000000       14072.000000
mean     2501.793277           6.147575
std     14791.558941           5.088043
min         0.000000           0.000000
25%        29.000000           3.000000
50%       164.000000           5.100000
75%       798.000000           8.100000
max    421670.000000          50.979000
Training model for Model_ID: Model_2_Low
            Area_Ha  Yield_QHa_Clipped
count  4.169800e+04       41698.000000
mean   9.459581e+03           8.838587
std    3.563016e+04           4.922655
min    2.000000e-02           0.000000
25%    7.700000e+01           5.600000
50%    6.035000e+02           8.100000
75%    4.138000e+03          11.000000
max    1.133397e+06          73.957000
Training model for Model_ID: Model_3_Medium
             Area_Ha  Yield_QHa_Clipped
count   46732.000000       46732.000000
mean    22512.366984          19.509123
std     55455.034767          14.87858

In [21]:
df.columns

Index(['State', 'District', 'Season', 'Year', 'Crop', 'Area_Ha', 'Yield_QHa',
       'Start_Year', 'District_norm', 'State_norm', 'latitude', 'longitude',
       'soil_ph', 'soil_oc', 'clay_pct', 'sand_pct', 'cec_cmol', 'avg_temp',
       'humidity_avg', 'rain_total', 'solar_avg', 'Yield_QHa_Clipped',
       'Model_ID'],
      dtype='object')

In [22]:
DROP_COLS = [
    "State",
    "District",
    "State_norm",
    "District_norm",
    "Year",
    "Yield_QHa",
    "Yield_QHa_Clipped",
    "Model_ID",
]

In [23]:
def tune_xgb(X_train, y_train, X_val, y_val):
    params = {
        "n_estimators": [300, 600, 900],
        "learning_rate": [0.05, 0.1, 0.2],
        "max_depth": [4, 6, 8],
        "min_child_weight": [1, 3, 5],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }

    base = XGBRegressor(objective="reg:squarederror", random_state=8)

    search = RandomizedSearchCV(
        estimator=base,
        param_distributions=params,
        n_iter=25,
        scoring="neg_median_absolute_error",
        cv=3,
        refit=True,
        n_jobs=-1,
        verbose=1,
        random_state=8,
    )

    search.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    return search.best_estimator_


In [24]:
os.makedirs("models", exist_ok=True)
for model_id in sorted(df["Model_ID"].unique()):
    data = df[df["Model_ID"] == model_id].copy()
    X = data.drop(columns=DROP_COLS)
    y = data["Yield_QHa_Clipped"]

    # split set
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=8
    )

    # preprocess: only one-hot for categorical, passthrough numerics
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ],
        remainder="passthrough",
    )

    # fit preprocessing
    X_train_p = preprocessor.fit_transform(X_train)
    X_test_p = preprocessor.transform(X_test)

    # tune + train model
    model = tune_xgb(X_train_p, y_train, X_test_p, y_test)

    # test predictions
    y_pred = model.predict(X_test_p)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"Model_ID {model_id} -> MSE: {mse:.4f}, R2: {r2:.4f}")

    # save model & preprocess together (deploy safely)
    joblib.dump(
        {"model": model, "preprocessor": preprocessor},
        f"models/model_{model_id}.joblib",
    )


Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model_ID Model_1_UltraLow -> MSE: 4.7657, R2: 0.8127
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model_ID Model_2_Low -> MSE: 7.2771, R2: 0.7017
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model_ID Model_3_Medium -> MSE: 38.3398, R2: 0.8282
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model_ID Model_4_HighVeg -> MSE: 939.9106, R2: 0.8581
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model_ID Model_5_UltraHigh -> MSE: 10890.6167, R2: 0.8908
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model_ID Model_6_Coconut -> MSE: 684094320.3766, R2: 0.6547
